# Chapter 7: Cleaning Messy Data

The programs from Chapter 7 of *Data Analysis with Python, Hands-On* by Aadhavan Muthurengan, in the book's order. Each code cell is one program; the same program is in this folder as a `.py` file, and its expected output is in the `.out` file with the same name.

Run the cells from top to bottom (**Run > Run All Cells**). The programs read the datasets from `../data/`, so keep this notebook in its chapter folder.

## Look Before You Clean

`inspect_raw.py`

In [1]:
import pandas as pd

raw = pd.read_csv("../data/orders_raw.csv")

print(raw.dtypes)
print()
print("missing values per column:")
print(raw.isna().sum()[raw.isna().sum() > 0])
print()
print("region spellings:")
print(raw["region"].value_counts().sort_index())
print("duplicate rows:", raw.duplicated().sum())
print("first prices:", raw["unit_price"].head(3).tolist())
print("first dates:", raw["order_date"].head(3).tolist())

order_id           str
order_date         str
customer_id        str
region             str
channel            str
category           str
product            str
quantity       float64
unit_price         str
discount       float64
dtype: object

missing values per column:
quantity    14
dtype: int64

region spellings:
region
East      51
East      10
North     71
North     10
South     31
South      6
West      42
West       6
east      22
north     24
south     13
west      17
Name: count, dtype: int64
duplicate rows: 3
first prices: ['$18.00', '$35.00', '$50.00']
first dates: ['05/01/2025', '2025-01-06', '2025-01-08']


## A Date Trap

`date_trap.py`

In [2]:
import pandas as pd

dates = pd.Series(["05/01/2025", "2025-01-06", "2025-01-08"])

guessed = pd.to_datetime(dates, format="mixed", dayfirst=True)
print(guessed.dt.strftime("%d %b %Y").tolist())

iso = pd.to_datetime(dates, format="%Y-%m-%d", errors="coerce")
dmy = pd.to_datetime(dates, format="%d/%m/%Y", errors="coerce")
print(iso.fillna(dmy).dt.strftime("%d %b %Y").tolist())

['05 Jan 2025', '01 Jun 2025', '01 Aug 2025']
['05 Jan 2025', '06 Jan 2025', '08 Jan 2025']


## Cleaning, Step by Step

`clean_orders.py`

In [3]:
import pandas as pd

raw = pd.read_csv("../data/orders_raw.csv")
clean = raw.drop_duplicates().copy()

clean["region"] = clean["region"].str.strip().str.title()
clean["unit_price"] = (
    clean["unit_price"].str.replace("$", "", regex=False).astype(float)
)
# Two date layouts: try each one explicitly, never guess
iso = pd.to_datetime(
    clean["order_date"], format="%Y-%m-%d", errors="coerce"
)
dmy = pd.to_datetime(
    clean["order_date"], format="%d/%m/%Y", errors="coerce"
)
clean["order_date"] = iso.fillna(dmy)
assert clean["order_date"].notna().all()

too_big = clean["quantity"] > 50
print("rows with an impossible quantity:", too_big.sum())
clean.loc[too_big, "quantity"] = None
clean["quantity"] = clean["quantity"].fillna(clean["quantity"].median())
clean["quantity"] = clean["quantity"].astype(int)

print("rows:", len(raw), "->", len(clean))
print("regions:", sorted(clean["region"].unique()))
print(clean.dtypes[["order_date", "quantity", "unit_price"]])
print(clean[["order_date", "region", "quantity", "unit_price"]].head(3))

rows with an impossible quantity: 1
rows: 303 -> 300
regions: ['East', 'North', 'South', 'West']
order_date    datetime64[us]
quantity               int64
unit_price           float64
dtype: object
  order_date region  quantity  unit_price
0 2025-01-05  South         1        18.0
1 2025-01-06   West         1        35.0
2 2025-01-08   East         1        50.0


## Validate With Checks

`validate.py`

In [4]:
import pandas as pd

orders = pd.read_csv("../data/orders.csv", parse_dates=["order_date"])

checks = {
    "order IDs are unique": orders["order_id"].is_unique,
    "no missing values": orders.notna().all().all(),
    "quantities between 1 and 20": orders["quantity"].between(1, 20).all(),
    "prices are positive": (orders["unit_price"] > 0).all(),
    "dates are in 2025": (orders["order_date"].dt.year == 2025).all(),
    "known regions only": orders["region"]
    .isin(["North", "South", "East", "West"])
    .all(),
}
for name, passed in checks.items():
    print(f"{'PASS' if passed else 'FAIL'}  {name}")

PASS  order IDs are unique
PASS  no missing values
PASS  quantities between 1 and 20
PASS  prices are positive
PASS  dates are in 2025
PASS  known regions only
